# CS781 Phase 7 - Prior Case Retrieval (PCR) with the Phase-1 models

One notebook for every Phase-7 experiment, built like the earlier phases. Set the experiment in **section 3** and run all, once per experiment.

| switch | options |
|---|---|
| `MODEL_TYPE` | `"gpt"`, `"modern"`, `"rnn"`, `"t5"` (T5: encoder only) - every case is encoded by the same Phase-1 model (bi-encoder) |
| `FT_METHOD` | `"full"`, `"partial"` (top layers), `"lorma"`, `"ia3"` (learned scaling of keys, values and FFN activations), `"prefix"` (prefix tuning, gpt / modern) |
| `USE_KD` | `True`: a smaller student (first `STUDENT_LAYERS` layers of `MODEL_TYPE`) learns the saved teacher's similarity scores |
| `CHUNK_MODE`, `N_CHUNKS` | which 512-token chunks of a long case are read: `"first"`, `"first_last"`, `"spread"` |
| `SIMILARITY` | `"cosine"` (case vector = mean of chunk vectors) or `"maxsim"` (every query chunk is matched to its best candidate chunk) |
| `NEGATIVES` | `"bm25"` (in-batch + one BM25 hard negative per pair), `"inbatch"` (in-batch only), `"random"` (in-batch + a random candidate) |

Pipeline: case (list of sentences) -> Phase-1 BPE -> `N_CHUNKS` chunks of 512 tokens -> Phase-1 model -> mean of token states per chunk -> case vector -> cosine similarity between the query and every candidate of the same split's pool -> top 7 -> `prediction.json`.

Training: contrastive loss (InfoNCE: the relevant candidate must score higher than the other candidates in the batch and a hard negative). Every run also prints a **BM25** baseline (word-matching retrieval, no model), the **zero-shot** Phase-1 model (no training), and a BM25 + model **fusion** whose weight is tuned on dev. Two submission files are written: model only, and fused.

## 1. Hugging Face Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("hf_token")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
  print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 3. Configuration

In [ ]:
import os
import random
import json
import time
import math
import zipfile
import gc
import torch

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_QUERY_FILE = "Phase_7_PCR/Train/train.json"
TRAIN_CAND_FILE = "Phase_7_PCR/Train/train_candidates.json"
DEV_QUERY_FILE = "Phase_7_PCR/Dev/dev.json"
DEV_CAND_FILE = "Phase_7_PCR/Dev/dev_candidates.json"
TEST_QUERY_FILE = "Phase_7_PCR/Test/test.json"
TEST_CAND_FILE = "Phase_7_PCR/Test/test_candidates.json"

# ---------------- the experiment ----------------
EXPERIMENT_NAME = "modern_full_fl2"    # <-- CHANGED for this run
MODEL_TYPE = "modern"    # <-- CHANGED for this run
FT_METHOD = "full"    # <-- CHANGED for this run
SMOKE_TEST = False
USE_PHASE1_WEIGHTS = True    # False = random initialisation (ablation: do the Phase-1 representations transfer?)

# long cases -> chunks -> one vector
CHUNK_TOKENS = 512
N_CHUNKS = 2                 # chunks per case
CHUNK_MODE = "first_last"    # "first" | "first_last" | "spread" (N_CHUNKS chunks spread evenly over the case)
TOKEN_POOL = "mean"          # chunk vector = "mean" of its token states | "last" token state
SIMILARITY = "cosine"        # "cosine" (mean of chunk vectors) | "maxsim" (each query chunk vs its best candidate chunk)
TEMPERATURE = 0.05           # similarity / temperature = logits of the contrastive loss

# training pairs and negatives
NEGATIVES = "bm25"           # "bm25" (in-batch + 1 BM25 hard negative) | "inbatch" | "random" (in-batch + 1 random candidate)
HARD_NEG_DEPTH = 30          # BM25 hard negatives are sampled from the top HARD_NEG_DEPTH non-relevant candidates
MAX_TRAIN_PAIRS = 6000       # (query, relevant candidate) pairs used for training (None = all)
TOP_K = 7                    # the evaluation looks at the first 7 candidates
FUSION = True                # also write a BM25 + model submission (weight tuned on dev)
ZERO_SHOT_EVAL = True        # score the untrained Phase-1 model on dev first
BM25_K1, BM25_B = 1.2, 0.75   # starting values; TUNE_BM25 picks the best pair on dev
TUNE_BM25 = True
SKIP_TRAINING = False        # True: use the zero-shot Phase-1 model as the final retriever (no training)

# knowledge distillation: student = first STUDENT_LAYERS layers of MODEL_TYPE, learns the teacher's similarity scores
USE_KD = False
STUDENT_LAYERS = 2
TEACHER_MODEL_TYPE = "modern"
TEACHER_FT_METHOD = "full"
TEACHER_PCR_PATH = "/kaggle/input/datasets/medinim/phase7-teacher/pcr_model.pt"
KD_TEMPERATURE = 2.0
KD_ALPHA = 0.5               # loss = KD_ALPHA * contrastive(gold) + (1 - KD_ALPHA) * T^2 * KL(teacher scores || student scores)

# LoRMA / IA3 / prefix / partial
ADAPTER_RANK = 16
ADAPTER_ALPHA = 32
PREFIX_LENGTH = 32
PREFIX_HIDDEN = 512
PARTIAL_TOP_LAYERS = 2

# Phase-1 models: one Kaggle dataset with model_<type>.safetensors and bpe_legal_<type>.json
PHASE1_FOLDER = "/kaggle/input/datasets/medinim/phase1-models"   # searched automatically below if this path is wrong
if not os.path.exists(os.path.join(PHASE1_FOLDER, "model_modern.safetensors")):
  import glob
  hits = glob.glob("/kaggle/input/**/model_modern.safetensors", recursive=True)
  if hits:
    PHASE1_FOLDER = os.path.dirname(hits[0])
print("Phase-1 models from:", PHASE1_FOLDER)
PHASE1_MODEL_FILES = {k: os.path.join(PHASE1_FOLDER, f"model_{k}.safetensors") for k in ["gpt", "modern", "rnn", "t5"]}
PHASE1_BPE_FILES = {k: os.path.join(PHASE1_FOLDER, f"bpe_legal_{k}.json") for k in ["gpt", "modern", "rnn", "t5"]}

# Phase-1 architecture sizes: must be EXACTLY what Phase 1 trained with
MODEL_CONTEXT_LENGTH = 2048
N_EMBD, N_LAYER, N_HEAD, N_KV_HEAD = 640, 10, 10, 5
D_FF = 4 * N_EMBD
RNN_CELL, RNN_HIDDEN, RNN_LAYERS = "lstm", 1152, 4
T5_D_MODEL, T5_ENC_LAYERS, T5_DEC_LAYERS, T5_HEADS, T5_D_FF, T5_N_SENTINELS = 512, 8, 8, 8, 2048, 100
DROPOUT = 0.1
LOAD_PCR_PATH = None         # path to a saved pcr_model.pt -> skip training, only retrieve ("auto" = find it in the inputs)
LOAD_FROM_EXPERIMENT = None  # with "auto": which trained experiment to reload (None = EXPERIMENT_NAME)
REQUIRE_GPU = True

# training (Trainer, like the earlier phases)
if FT_METHOD in ("full", "partial"):
  LEARNING_RATE = 5e-5
elif FT_METHOD == "ia3":
  LEARNING_RATE = 3e-3       # IA3 trains only scaling vectors (Liu et al. 2022 use a large learning rate)
else:
  LEARNING_RATE = 5e-4       # LoRMA / prefix
if MODEL_TYPE == "rnn":
  LEARNING_RATE = 5e-4
if MODEL_TYPE == "t5":
  LEARNING_RATE = 1e-3 if FT_METHOD != "full" else 3e-4
LR_OVERRIDE = None
if LR_OVERRIDE:
  LEARNING_RATE = LR_OVERRIDE
EPOCHS = 2
BATCH_SIZE = 2               # (query, positive, negative) triples per GPU; every case = N_CHUNKS sequences of 512 tokens
GRAD_ACCUM = 4               # 2 GPUs x 2 x 4 = 16 triples per update
ENCODE_BATCH = 16            # cases per forward pass when encoding a candidate pool
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if SMOKE_TEST:
  EPOCHS = 1
  MAX_TRAIN_PAIRS = 64
if REQUIRE_GPU and not torch.cuda.is_available():
  raise RuntimeError("No GPU! In Kaggle: Settings -> Accelerator -> GPU T4 x2, then Save & Run All again.")
if USE_KD and not os.path.exists(TEACHER_PCR_PATH):
  import glob
  hits = glob.glob("/kaggle/input/**/pcr_model.pt", recursive=True)
  if hits:
    TEACHER_PCR_PATH = hits[0]
  print("KD teacher:", TEACHER_PCR_PATH)

if LOAD_PCR_PATH == "auto":
  import glob
  src = LOAD_FROM_EXPERIMENT or EXPERIMENT_NAME
  hits = glob.glob(f"/kaggle/input/**/{src}/pcr_model.pt", recursive=True) + glob.glob(f"/kaggle/input/**/{src}.pt", recursive=True)
  if not hits:
    raise FileNotFoundError(f"no {src}/pcr_model.pt or {src}.pt in the inputs: attach the output (or a dataset) that contains it")
  LOAD_PCR_PATH = hits[0]
  print("retrieve only, using", LOAD_PCR_PATH)

OUTPUT_DIR = "/kaggle/working/cs781_phase7"
EXP_DIR = os.path.join(OUTPUT_DIR, EXPERIMENT_NAME)
os.makedirs(EXP_DIR, exist_ok=True)
RESULTS_PATH = os.path.join(OUTPUT_DIR, "results.jsonl")
print("Experiment:", EXPERIMENT_NAME, "|", MODEL_TYPE, FT_METHOD, f"| {N_CHUNKS} x {CHUNK_TOKENS} tokens, {CHUNK_MODE}, {SIMILARITY}, negatives: {NEGATIVES}", "| KD" if USE_KD else "")

## 4. Tokenizer (the Phase-1 BPE, unchanged)

In [ ]:
import json
import heapq
from collections import Counter, defaultdict

SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
WORD_END = "</w>"

class ByteLevelBPE:
  def __init__(self, vocab_size):
      self.vocab_size = vocab_size
      self.merges = []
      self.merge_ranks = {}
      self.itos = []
      self.stoi = {}
      self._cache = {}
      self.pad_id = None
      self.bos_id = None
      self.eos_id = None

  def _word_to_symbols(self, word):
      byte_vals = word.encode("utf-8")
      symbols = [chr(b) for b in byte_vals]
      symbols.append(WORD_END)
      return symbols

  def train(self, sentences):
      word_freq = Counter()
      for sent in sentences:
          for word in sent.split():
              word_freq[word] += 1

      words = list(word_freq)
      freqs = [word_freq[w] for w in words]
      word_symbols = [self._word_to_symbols(w) for w in words]

      vocab_list = list(SPECIALS)
      vocab_seen = set(vocab_list)

      initial_symbols = {chr(b) for b in range(256)} | {WORD_END}

      for s in sorted(initial_symbols):
          if s not in vocab_seen:
              vocab_list.append(s)
              vocab_seen.add(s)

      # this loop recounts every pair every merge, which is far too slow
      # for the legal corpus. Replace it with incremental pair counts + pair_to_words.
      pair_counts = Counter()
      pair_to_words = defaultdict(set)
      for wi, symbols in enumerate(word_symbols):
          f = freqs[wi]
          for i in range(len(symbols) - 1):
              p = (symbols[i], symbols[i + 1])
              pair_counts[p] += f
              pair_to_words[p].add(wi)

      heap = [(-c, p) for p, c in pair_counts.items()]
      heapq.heapify(heap)

      while len(vocab_list) < self.vocab_size:
          best_pair, best_count = None, 0
          while heap:
              negc, p = heapq.heappop(heap)
              if pair_counts.get(p, 0) == -negc and -negc > 0:
                  best_pair, best_count = p, -negc
                  break
          if best_pair is None or best_count <= 1:
              break

          a, b = best_pair
          merged = a + b
          changed = set()

          for wi in list(pair_to_words[best_pair]):
              symbols = word_symbols[wi]
              f = freqs[wi]

              old_pairs = Counter(zip(symbols, symbols[1:]))   # positions, so overlaps count
              new_symbols = []
              i = 0
              while i < len(symbols):
                  if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                      new_symbols.append(merged)
                      i += 2
                  else:
                      new_symbols.append(symbols[i])
                      i += 1
              word_symbols[wi] = new_symbols
              new_pairs = Counter(zip(new_symbols, new_symbols[1:]))

              for p, c in old_pairs.items():
                  pair_counts[p] -= c * f
                  changed.add(p)
                  if p not in new_pairs:
                      pair_to_words[p].discard(wi)
              for p, c in new_pairs.items():
                  pair_counts[p] += c * f
                  changed.add(p)
                  pair_to_words[p].add(wi)

          for p in changed:
              c = pair_counts[p]
              if c > 0:
                  heapq.heappush(heap, (-c, p))
              else:
                  del pair_counts[p]
                  pair_to_words.pop(p, None)

          self.merges.append([a, b])
          self.merge_ranks[(a, b)] = len(self.merges) - 1

          if merged not in vocab_seen:
              vocab_list.append(merged)
              vocab_seen.add(merged)

      self.itos = vocab_list
      self.stoi = {tok: i for i, tok in enumerate(self.itos)}
      self.pad_id = self.stoi["<pad>"]
      self.bos_id = self.stoi["<s>"]
      self.eos_id = self.stoi["</s>"]
      return self

  def encode_word(self, word):
      if word in self._cache:
        return self._cache[word]
      symbols = self._word_to_symbols(word)
      while len(symbols) > 1:
          pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
          ranked = [(self.merge_ranks[p], i) for i, p in enumerate(pairs) if p in self.merge_ranks]
          if not ranked:
              break
          _, i = min(ranked)
          symbols = symbols[:i] + [symbols[i] + symbols[i + 1]] + symbols[i + 2:]

      self._cache[word] = symbols
      return symbols

  def encode_ids(self, sentence):
      unk = self.stoi["<unk>"]
      return [self.stoi.get(t, unk) for t in self.encode(sentence)]

  def decode(self, ids):
      text = "".join(self.itos[i] for i in ids if self.itos[i] not in SPECIALS)
      words = text.split(WORD_END)
      out = [bytes(ord(c) for c in w).decode("utf-8", errors="replace") for w in words if w]
      return " ".join(out)

  def encode(self, sentence):
      tokens = []
      for word in sentence.split():
          tokens.extend(self.encode_word(word))
      return tokens

  def save(self, path):
      with open(path, "w", encoding="utf-8") as f:
          json.dump({"merges": self.merges, "itos": self.itos,
                      "vocab_size": self.vocab_size}, f, ensure_ascii=False)

  @staticmethod
  def load_bpe(path):
      with open(path, "r", encoding="utf-8") as f:
          data = json.load(f)
      bpe = ByteLevelBPE(vocab_size=data["vocab_size"])
      bpe.merges = data["merges"]
      bpe.itos = data["itos"]
      bpe.stoi = {tok: i for i, tok in enumerate(bpe.itos)}
      bpe.merge_ranks = {(a, b): i for i, (a, b) in enumerate(bpe.merges)}
      bpe.pad_id = bpe.stoi["<pad>"]
      bpe.bos_id = bpe.stoi["<s>"]
      bpe.eos_id = bpe.stoi["</s>"]
      return bpe

In [ ]:
import hashlib
for k, f in PHASE1_BPE_FILES.items():   # all four must be the same tokenizer (needed for KD)
  print(k, hashlib.md5(open(f, "rb").read()).hexdigest())
bpe = ByteLevelBPE.load_bpe(PHASE1_BPE_FILES[MODEL_TYPE])
UNK_ID = bpe.stoi["<unk>"]
print("vocab size:", len(bpe.itos))

## 5. Download the Phase-7 data

Every split has its own candidate pool: train queries are ranked against `train_candidates.json`, dev against `dev_candidates.json`, test against `test_candidates.json`. A case's `text` is a list of sentences; it is joined, tokenized once, and cut into `N_CHUNKS` chunks of 512 tokens (`CHUNK_MODE`). A query is never returned as its own candidate.

In [ ]:
from huggingface_hub import hf_hub_download
from collections import Counter
import re
import numpy as np

def load_json_file(fname):
  p = hf_hub_download(repo_id=HF_DATASET_REPO, filename=fname, repo_type="dataset", token=HF_TOKEN)
  with open(p, "r", encoding="utf-8") as f:
    return json.load(f)

def as_text(x):
  if isinstance(x, list):
    return " ".join(as_text(t) for t in x)
  if isinstance(x, dict):
    return as_text(x.get("text", ""))
  return str(x) if x is not None else ""

def to_records(data):
  """accepts a list of dicts, or {id: dict / text}, or {"data": [...]}; returns a list of dicts with an "id" """
  if isinstance(data, dict):
    for key in ("data", "queries", "candidates"):
      if key in data and isinstance(data[key], list):
        return to_records(data[key])
    out = []
    for k, v in data.items():
      v = dict(v) if isinstance(v, dict) else {"text": v}
      v.setdefault("id", k)
      out.append(v)
    return out
  out = []
  for d in data:
    d = dict(d)
    for key in ("id", "query_id", "candidate_id", "case_id", "doc_id"):
      if key in d:
        d["id"] = d[key]
        break
    out.append(d)
  return out

def read_split(qfile, cfile, with_labels):
  queries, pool = [], []
  for d in to_records(load_json_file(qfile)):
    rel = None
    if with_labels:
      for key in ("relevant_candidates", "relevant", "labels", "positives"):
        if key in d:
          rel = [str(x) for x in d[key]]
          break
      if rel is None:
        raise ValueError(f"no relevant_candidates in query {d.get('id')}; keys: {list(d.keys())}")
    queries.append({"id": str(d["id"]), "text": as_text(d.get("text", "")), "relevant": rel})
  for d in to_records(load_json_file(cfile)):
    pool.append({"id": str(d["id"]), "text": as_text(d.get("text", ""))})
  return queries, pool

train_q, train_pool = read_split(TRAIN_QUERY_FILE, TRAIN_CAND_FILE, True)
dev_q, dev_pool = read_split(DEV_QUERY_FILE, DEV_CAND_FILE, True)
test_q, test_pool = read_split(TEST_QUERY_FILE, TEST_CAND_FILE, False)
for name, qs, pool in [("train", train_q, train_pool), ("dev", dev_q, dev_pool), ("test", test_q, test_pool)]:
  ids = set(c["id"] for c in pool)
  msg = f"{name}: {len(qs)} queries | candidate pool {len(pool)}"
  if qs[0]["relevant"] is not None:
    g = [len(q["relevant"]) for q in qs]
    inside = sum(1 for q in qs for r in q["relevant"] if r in ids) / max(1, sum(g))
    msg += f" | relevant per query: mean {np.mean(g):.1f}, median {int(np.median(g))}, max {max(g)} | relevant ids found in the pool: {100 * inside:.1f}%"
    msg += f" | best possible F1@7: {2 * sum(min(TOP_K, x) for x in g) / (TOP_K * len(g) + sum(g)):.3f}"
  print(msg)
  w = sorted(len(d["text"].split()) for d in qs + pool)
  print(f"   words per case: median {w[len(w) // 2]}, 90% {w[int(0.9 * len(w))]}, max {w[-1]}")

def chunk_starts(n_tokens):
  C = CHUNK_TOKENS
  if n_tokens <= C:
    return [0]
  if CHUNK_MODE == "first":
    return [s for s in range(0, n_tokens, C)][:N_CHUNKS]
  if CHUNK_MODE == "first_last":
    first = list(range(0, n_tokens, C))[:(N_CHUNKS + 1) // 2]
    last = [max(0, n_tokens - C * (i + 1)) for i in range(N_CHUNKS // 2)][::-1]
    out = []
    for s in first + last:
      if s not in out:
        out.append(s)
    return out
  if CHUNK_MODE == "spread":
    if N_CHUNKS == 1:
      return [0]
    return sorted(set(int(round(i * (n_tokens - C) / (N_CHUNKS - 1))) for i in range(N_CHUNKS)))
  raise ValueError(CHUNK_MODE)

# one table of all cases (queries and candidates of all splits); a case is referred to by its row number
DOCS = []          # list of chunk lists
DOC_LEN = []       # tokens per case
def add_case(d):
  ids = bpe.encode_ids(d["text"])
  d["row"] = len(DOCS)
  DOCS.append([ids[s:s + CHUNK_TOKENS] for s in chunk_starts(len(ids))])
  DOC_LEN.append(len(ids))

t0 = time.time()
for d in train_q + train_pool + dev_q + dev_pool + test_q + test_pool:
  add_case(d)
bpe._cache.clear()
print(f"tokenized {len(DOCS)} cases in {time.time() - t0:.0f}s | tokens per case: median {int(np.median(DOC_LEN))}, 90% {int(np.percentile(DOC_LEN, 90))}")
print(f"the {N_CHUNKS} chunks ({CHUNK_MODE}) cover on average {100 * np.mean([min(1, N_CHUNKS * CHUNK_TOKENS / max(1, n)) for n in DOC_LEN]):.0f}% of a case")

## 6. BM25 baseline and hard negatives (no neural model)

BM25 (Robertson et al.) is the classic word-matching ranking function: a candidate scores high when it contains the query's words, words that are rare in the pool count more (idf), and long candidates are normalised (`b`). It is the standard strong baseline for prior-case retrieval.

It is used for: (1) a baseline score, (2) **hard negatives** for training (candidates that look similar to the query by words but are not relevant), (3) the optional BM25 + model fusion.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import scipy.sparse as sp

def bm25_scores(queries, pool):
  """returns a (n_queries, n_pool) array of BM25 scores; the vocabulary and idf come from this split's pool"""
  vec = CountVectorizer(lowercase=True, token_pattern=r"(?u)\b[a-z][a-z0-9]+\b", stop_words="english", min_df=2, max_df=0.9)
  X = vec.fit_transform([c["text"] for c in pool]).tocsr().astype(np.float32)      # pool x vocabulary counts
  N = X.shape[0]
  df = np.bincount(X.indices, minlength=X.shape[1])
  idf = np.log((N - df + 0.5) / (df + 0.5) + 1.0)
  dl = np.asarray(X.sum(axis=1)).ravel()
  avgdl = dl.mean()
  X = X.tocoo()
  tf = X.data
  w = idf[X.col] * tf * (BM25_K1 + 1) / (tf + BM25_K1 * (1 - BM25_B + BM25_B * dl[X.row] / avgdl))
  W = sp.csr_matrix((w, (X.row, X.col)), shape=X.shape)
  Q = vec.transform([q["text"] for q in queries]).astype(np.float32)
  Q.data = 1 + np.log(Q.data)                            # query term weight: 1 + log(count) (long queries)
  return (Q @ W.T).toarray()

def self_mask(queries, pool):
  pos = {c["id"]: j for j, c in enumerate(pool)}
  m = np.zeros((len(queries), len(pool)), dtype=bool)
  for i, q in enumerate(queries):
    if q["id"] in pos:
      m[i, pos[q["id"]]] = True                           # a case is never its own prior case
  return m

def rank(scores, queries, pool):
  s = np.where(self_mask(queries, pool), -1e9, scores)
  return np.argsort(-s, axis=1)

def evaluate(order, queries, pool, ks=(TOP_K,)):
  """micro precision / recall / F1 at k over all queries (the competition metric is micro-F1@7)"""
  out = {}
  for k in ks:
    hits = gold = 0
    for i, q in enumerate(queries):
      rel = set(q["relevant"])
      top = [pool[j]["id"] for j in order[i, :k]]
      hits += sum(1 for c in top if c in rel)
      gold += len(rel)
    p = hits / (k * len(queries))
    r = hits / max(1, gold)
    out[k] = {"P": p, "R": r, "F1": 2 * p * r / (p + r) if p + r else 0.0}
  return out

def show(name, ev, k=TOP_K):
  e = ev[k]
  print(f"{name:42s} F1@{k} {e['F1']:.4f} | P@{k} {e['P']:.4f} | R@{k} {e['R']:.4f}")

if TUNE_BM25:
  t0 = time.time()
  best = (-1, BM25_K1, BM25_B)
  grid = {}
  for k1 in (0.6, 1.2, 2.0, 3.0):
    for b in (0.3, 0.6, 0.75, 0.9):
      BM25_K1, BM25_B = k1, b
      f1 = evaluate(rank(bm25_scores(dev_q, dev_pool), dev_q, dev_pool), dev_q, dev_pool)[TOP_K]["F1"]
      grid[(k1, b)] = round(f1, 4)
      if f1 > best[0]:
        best = (f1, k1, b)
  _, BM25_K1, BM25_B = best
  print(f"BM25 tuned on dev in {time.time() - t0:.0f}s: k1 = {BM25_K1}, b = {BM25_B} | F1@7 by (k1, b): {grid}")

t0 = time.time()
bm25_train = bm25_scores(train_q, train_pool)
bm25_dev = bm25_scores(dev_q, dev_pool)
bm25_test = bm25_scores(test_q, test_pool)
print(f"BM25 for all splits in {time.time() - t0:.0f}s")
ev_bm25 = evaluate(rank(bm25_dev, dev_q, dev_pool), dev_q, dev_pool, ks=(1, 3, 5, 7, 10, 20, 50, 100))
show("DEV  BM25 (no model)", ev_bm25)
print("   BM25 recall@50 / @100 (how many relevant cases are in the top 50 / 100):",
      round(ev_bm25[50]["R"], 3), "/", round(ev_bm25[100]["R"], 3))

## 7. Training triples (query, relevant candidate, negative)

Every (query, relevant candidate) pair of the training split becomes one example. Its negative is a BM25 hard negative (sampled from the top `HARD_NEG_DEPTH` non-relevant candidates), a random candidate, or none (`"inbatch"`). In the loss, every other candidate in the batch is a negative too (in-batch negatives). The dev triples give the validation loss for checkpoint selection.

In [ ]:
from datasets import Dataset

def make_triples(queries, pool, bm25, max_pairs, seed):
  rng = random.Random(seed)
  pos_of = {c["id"]: j for j, c in enumerate(pool)}
  order = rank(bm25, queries, pool)
  rows = {"q": [], "p": [], "n": []}
  for i, q in enumerate(queries):
    rel = set(q["relevant"])
    hard = [j for j in order[i, :HARD_NEG_DEPTH + len(rel)] if pool[j]["id"] not in rel][:HARD_NEG_DEPTH]
    for r in q["relevant"]:
      if r not in pos_of:
        continue
      if NEGATIVES == "bm25" and hard:
        n = rng.choice(hard)
      else:
        n = rng.randrange(len(pool))
        while pool[n]["id"] in rel or pool[n]["id"] == q["id"]:
          n = rng.randrange(len(pool))
      rows["q"].append(q["row"])
      rows["p"].append(pool[pos_of[r]]["row"])
      rows["n"].append(pool[n]["row"])
  idx = list(range(len(rows["q"])))
  rng.shuffle(idx)
  if max_pairs is not None:
    idx = idx[:max_pairs]
  return Dataset.from_dict({k: [v[i] for i in idx] for k, v in rows.items()})

train_dataset = make_triples(train_q, train_pool, bm25_train, MAX_TRAIN_PAIRS, SEED)
val_dataset = make_triples(dev_q, dev_pool, bm25_dev, 400, SEED + 1)
print(f"training triples: {len(train_dataset)} | validation triples: {len(val_dataset)} | negatives: {NEGATIVES}")

def doc_tensor(rows):
  """(B, N_CHUNKS, CHUNK_TOKENS) token ids, right padded; a missing chunk is all padding"""
  t = torch.full((len(rows), N_CHUNKS, CHUNK_TOKENS), bpe.pad_id, dtype=torch.long)
  for b, r in enumerate(rows):
    for c, ids in enumerate(DOCS[r][:N_CHUNKS]):
      t[b, c, :len(ids)] = torch.tensor(ids, dtype=torch.long)
  return t

def data_collator(features):
  batch = {"q_ids": doc_tensor([f["q"] for f in features]), "p_ids": doc_tensor([f["p"] for f in features])}
  if NEGATIVES != "inbatch":
    batch["n_ids"] = doc_tensor([f["n"] for f in features])
  return batch

## 8. Phase-1 model code (copied unchanged from the Phase-1 notebook)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    """Module 2.3:  RMS(x) = sqrt( (1/E) * sum_i x_i^2 + eps ),   RMSNorm(x)_i = gamma_i * x_i / RMS(x)"""
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))   # learnable, no bias (beta)

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.gamma * (x / rms)


# ---- RoPE (Module 2.3 / 3.1, Assignment 3 "rope.py") ----
def build_rope_cache(max_seq_len, d_head, base=10000.0):
    # theta_i = base^(-2(i-1)/d) for every pair of dimensions; angle = position m * theta_i
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))   # (d_head/2,)
    positions = torch.arange(max_seq_len).float()
    freqs = torch.outer(positions, inv_freq)                                  # (T, d_head/2)
    angles = torch.cat([freqs, freqs], dim=-1)                                # (T, d_head)
    return angles.cos(), angles.sin()


def rotate_half(x):
    # split the last dim in half and return (-x2, x1)
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    # x: (B, n_heads, T, d_head); cos, sin: (B, 1, T, d_head)
    return x * cos + rotate_half(x) * sin


class CausalSelfAttention(nn.Module):
    """Masked multi-head self-attention (Module 2.3, Assignment 3).
    n_kv_heads == n_heads -> normal MHA,  n_kv_heads < n_heads -> grouped-query attention (GQA).
    past_kv = (k, v) of earlier tokens -> KV cache for generation (Module 2.3).
    attention_mask (B, S): 1 = real token, 0 = padding (Module 2.3: masks prevent attention over padding)."""
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.1, use_rope=False):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        self.attention_scale = 1.0 / math.sqrt(self.d_head)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)
        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, position_ids, attention_mask=None, past_kv=None):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            cos = self.rope_cos[position_ids].unsqueeze(1)   # (B, 1, T, d_head)
            sin = self.rope_sin[position_ids].unsqueeze(1)
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)

        if past_kv is not None:                  # KV cache: reuse keys/values of earlier tokens
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        new_kv = (k, v)

        if self.n_rep > 1:                       # GQA: each k/v head is shared by n_rep query heads
            k = torch.repeat_interleave(k, self.n_rep, dim=1)
            v = torch.repeat_interleave(v, self.n_rep, dim=1)

        S = k.size(2)                             # number of keys = cached + new
        scores = (q @ k.transpose(-2, -1)) * self.attention_scale        # (B, H, T, S)
        allowed = self.causal_mask[S - T:S, :S].unsqueeze(0).unsqueeze(0)  # (1, 1, T, S)
        if attention_mask is not None:
            allowed = allowed & attention_mask[:, None, None, :].bool()
            # a left-padding position has no real token before it, so let every
            # position always see itself (otherwise its whole row would be masked)
            itself = torch.eye(S, dtype=torch.bool, device=x.device)[S - T:S, :S]
            allowed = allowed | itself.unsqueeze(0).unsqueeze(0)
        scores = scores.masked_fill(~allowed, float("-inf"))
        att = F.softmax(scores, dim=-1)
        att = self.attn_dropout(att)
        out = att @ v                                                      # (B, H, T, d_head)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.out_proj(out))
        return out, new_kv


class FeedForward(nn.Module):
    """Assignment 3: two linear layers with a GELU in between, plus dropout."""
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class SwiGLUFeedForward(nn.Module):
    """Module 3.1 names SwiGLU as the modern feed-forward activation:
    FFN(x) = W2( SiLU(W1 x) * (W3 x) ),  SiLU(a) = a * sigmoid(a)"""
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w3 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        a = self.w1(x)
        return self.dropout(self.w2(a * torch.sigmoid(a) * self.w3(x)))

In [ ]:
from dataclasses import dataclass


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 640
    n_layers: int = 10
    n_heads: int = 10
    n_kv_heads: int = 10            # == n_heads for plain MHA
    d_ff: int = 2560
    max_seq_len: int = 2048
    dropout: float = 0.1
    pos_emb_type: str = "learned"   # "learned" (GPT-2 / BERT, Module 2.3) or "rope"
    norm_type: str = "layernorm"    # "layernorm" or "rmsnorm"
    ffn_type: str = "gelu"          # "gelu" or "swiglu"


class Block(nn.Module):
    """Pre-norm block (Module 2.3, used by GPT-2 and modern LLMs):  x + Attn(Norm(x)),  then  x + FFN(Norm(x))"""
    def __init__(self, cfg):
        super().__init__()
        if cfg.norm_type == "layernorm":
            self.ln1 = nn.LayerNorm(cfg.d_model)
            self.ln2 = nn.LayerNorm(cfg.d_model)
        else:
            self.ln1 = RMSNorm(cfg.d_model)
            self.ln2 = RMSNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"))
        if cfg.ffn_type == "gelu":
            self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        else:
            self.ffn = SwiGLUFeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x, position_ids, attention_mask=None, past_kv=None):
        a, new_kv = self.attn(self.ln1(x), position_ids, attention_mask, past_kv)
        x = x + a
        x = x + self.ffn(self.ln2(x))
        return x, new_kv


class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "learned":
            self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)    # learned absolute positions
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        if cfg.norm_type == "layernorm":
            self.ln_f = nn.LayerNorm(cfg.d_model)
        else:
            self.ln_f = RMSNorm(cfg.d_model)
        # Module 2.1 "Proper Weight Initialization": Xavier/Glorot for the embedding matrices.
        # PyTorch's default N(0, 1) embedding + weight tying gives huge first logits (start loss ~80)
        nn.init.xavier_uniform_(self.tok_emb.weight)
        if cfg.pos_emb_type == "learned":
            nn.init.xavier_uniform_(self.pos_emb.weight)

    def forward(self, input_ids, labels=None, attention_mask=None, position_ids=None,
                past_key_values=None, use_cache=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        if position_ids is None:
            position_ids = torch.arange(past_len, past_len + T, device=input_ids.device).unsqueeze(0).expand(B, T)

        x = self.tok_emb(input_ids)
        if self.cfg.pos_emb_type == "learned":
            x = x + self.pos_emb(position_ids)
        x = self.drop(x)

        new_past = []
        for i, block in enumerate(self.blocks):
            layer_past = None if past_key_values is None else past_key_values[i]
            x, kv = block(x, position_ids, attention_mask, layer_past)
            new_past.append(kv)
        x = self.ln_f(x)

        if use_cache:
            x = x[:, -1:, :]              # generation only needs the prediction for the last position
        logits = x @ self.tok_emb.weight.t()   # weight tying

        loss = None
        if labels is not None:
            # predict token t+1 from positions <= t (labels are shifted inside, like GPT-2)
            loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, logits.size(-1)).float(),
                                   labels[:, 1:].reshape(-1))
        if use_cache:
            return {"loss": loss, "logits": logits, "past_key_values": new_past}
        return {"loss": loss, "logits": logits}

In [ ]:
class LSTMCell(nn.Module):
    """One time step of an LSTM (Assignment 2): packed linear layers compute f, i, g, o at once."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.x2g = nn.Linear(input_size, 4 * hidden_size)
        self.h2g = nn.Linear(hidden_size, 4 * hidden_size)
        with torch.no_grad():
            self.x2g.bias[0:hidden_size] = 1.0   # forget gate bias close to 1 (Module 2.1)
            self.h2g.bias[0:hidden_size] = 0.0

    def forward(self, x_t, state):
        h_prev, c_prev = state
        gates = self.x2g(x_t) + self.h2g(h_prev)
        f_pre, i_pre, g_pre, o_pre = gates.chunk(4, dim=-1)
        f_t = torch.sigmoid(f_pre)
        i_t = torch.sigmoid(i_pre)
        g_t = torch.tanh(g_pre)
        o_t = torch.sigmoid(o_pre)
        c_t = f_t * c_prev + i_t * g_t
        h_t = o_t * torch.tanh(c_t)
        return (h_t, c_t)


class GRUCell(nn.Module):
    """One time step of a GRU with the Module 2.1 equations (layer names as in Assignment 2)."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.x2z = nn.Linear(input_size, hidden_size)
        self.h2z = nn.Linear(hidden_size, hidden_size, bias=False)
        self.x2r = nn.Linear(input_size, hidden_size)
        self.h2r = nn.Linear(hidden_size, hidden_size, bias=False)
        self.x2n = nn.Linear(input_size, hidden_size)
        self.h2n = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x_t, h_prev):
        z_t = torch.sigmoid(self.x2z(x_t) + self.h2z(h_prev))          # update gate
        r_t = torch.sigmoid(self.x2r(x_t) + self.h2r(h_prev))          # reset gate
        n_t = torch.tanh(self.x2n(x_t) + self.h2n(r_t * h_prev))       # candidate h~
        h_t = (1 - z_t) * h_prev + z_t * n_t
        return h_t


class RNNLM(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers, cell_type="lstm", dropout=0.1):
        super().__init__()
        self.is_lstm = (cell_type == "lstm")
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.tok_emb = nn.Embedding(vocab_size, hidden_size)
        cell_cls = LSTMCell if self.is_lstm else GRUCell
        self.cells = nn.ModuleList([cell_cls(hidden_size, hidden_size) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.tok_emb.weight)     # Module 2.1: Xavier/Glorot initialisation

    def forward(self, input_ids, labels=None, attention_mask=None, past_key_values=None, use_cache=False):
        """past_key_values here = list of per-layer states (h, c) / h, carried during generation.
        attention_mask (B, T): at padding positions the state is NOT updated."""
        B, T = input_ids.shape
        x = self.dropout(self.tok_emb(input_ids))
        new_states = []
        for l in range(self.num_layers):
            cell = self.cells[l]
            if past_key_values is not None:
                state = past_key_values[l]
            else:
                h = torch.zeros(B, self.hidden_size, device=x.device, dtype=x.dtype)
                state = (h, torch.zeros_like(h)) if self.is_lstm else h
            outputs = []
            for t in range(T):           # explicit unroll over time (BPTT through these steps)
                new_state = cell(x[:, t, :], state)
                if attention_mask is not None:
                    m = attention_mask[:, t].unsqueeze(1).to(x.dtype)     # 1 = real token, 0 = pad
                    if self.is_lstm:
                        new_state = (m * new_state[0] + (1 - m) * state[0], m * new_state[1] + (1 - m) * state[1])
                    else:
                        new_state = m * new_state + (1 - m) * state
                state = new_state
                outputs.append(state[0] if self.is_lstm else state)
            new_states.append(state)
            x = self.dropout(torch.stack(outputs, dim=1))    # input of the next layer

        if use_cache:
            x = x[:, -1:, :]
        logits = x @ self.tok_emb.weight.t()       # weight tying (Assignment 3)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, logits.size(-1)).float(),
                                   labels[:, 1:].reshape(-1))
        if use_cache:
            return {"loss": loss, "logits": logits, "past_key_values": new_states}
        return {"loss": loss, "logits": logits}

In [ ]:
class T5RelativePositionBias(nn.Module):
    def __init__(self, n_heads, bidirectional, num_buckets=32, max_distance=128):
        super().__init__()
        self.bidirectional = bidirectional
        self.num_buckets = num_buckets
        self.max_distance = max_distance
        self.rel_emb = nn.Embedding(num_buckets, n_heads)     # one scalar per (bucket, head)

    def bucket(self, relative_position):
        # relative_position = key_pos - query_pos
        n = -relative_position
        num_buckets = self.num_buckets
        ret = torch.zeros_like(n)
        if self.bidirectional:                 # encoder: half the buckets for keys on the right
            num_buckets = num_buckets // 2
            ret = ret + (n < 0).long() * num_buckets
            n = torch.abs(n)
        else:                                  # decoder: only keys on the left exist
            n = torch.clamp(n, min=0)
        max_exact = num_buckets // 2           # small offsets get their own bucket
        is_small = n < max_exact
        # bigger offsets: logarithmically wider buckets, everything beyond max_distance shares the last one
        large = max_exact + (torch.log(torch.clamp(n, min=1).float() / max_exact)
                             / math.log(self.max_distance / max_exact)
                             * (num_buckets - max_exact)).long()
        large = torch.clamp(large, max=num_buckets - 1)
        return ret + torch.where(is_small, n, large)

    def forward(self, q_len, k_len, device):
        q_pos = torch.arange(q_len, device=device)[:, None]
        k_pos = torch.arange(k_len, device=device)[None, :]
        buckets = self.bucket(k_pos - q_pos)               # (q, k)
        bias = self.rel_emb(buckets)                       # (q, k, H)
        return bias.permute(2, 0, 1).unsqueeze(0)          # (1, H, q, k)


class T5Attention(nn.Module):
    """Self-attention (x_kv = x_q) or cross-attention (x_kv = encoder output)."""
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.attention_scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)

    def forward(self, x_q, x_kv, position_bias=None, allowed=None):
        B, Tq, C = x_q.shape
        Tk = x_kv.size(1)
        q = self.q_proj(x_q).view(B, Tq, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x_kv).view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x_kv).view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) * self.attention_scale
        if position_bias is not None:
            scores = scores + position_bias
        if allowed is not None:
            scores = scores.masked_fill(~allowed, float("-inf"))
        att = self.attn_dropout(F.softmax(scores, dim=-1))
        out = (att @ v).transpose(1, 2).contiguous().view(B, Tq, C)
        return self.out_proj(out)


class T5FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(F.relu(self.fc1(x))))   # dropout inside the FFN


class T5EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.self_attn = T5Attention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = T5FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, position_bias, src_allowed):
        h = self.norm1(x)
        x = x + self.dropout(self.self_attn(h, h, position_bias, src_allowed))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x


class T5DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.self_attn = T5Attention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.cross_attn = T5Attention(d_model, n_heads, dropout)
        self.norm3 = RMSNorm(d_model)
        self.ffn = T5FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, position_bias, causal_allowed, src_allowed):
        h = self.norm1(x)
        x = x + self.dropout(self.self_attn(h, h, position_bias, causal_allowed))   # masked self-attention
        h = self.norm2(x)
        x = x + self.dropout(self.cross_attn(h, enc_out, None, src_allowed))        # cross-attention
        x = x + self.dropout(self.ffn(self.norm3(x)))
        return x


class T5Model(nn.Module):
    def __init__(self, vocab_size, pad_id, d_model, n_enc_layers, n_dec_layers, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.pad_id = pad_id
        self.shared_emb = nn.Embedding(vocab_size, d_model)    # encoder input, decoder input and output
        self.dropout = nn.Dropout(dropout)
        self.enc_layers = nn.ModuleList([T5EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_enc_layers)])
        self.dec_layers = nn.ModuleList([T5DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_dec_layers)])
        self.enc_norm = RMSNorm(d_model)
        self.dec_norm = RMSNorm(d_model)
        self.enc_pos_bias = T5RelativePositionBias(n_heads, bidirectional=True)
        self.dec_pos_bias = T5RelativePositionBias(n_heads, bidirectional=False)
        nn.init.xavier_uniform_(self.shared_emb.weight)  # Module 2.1: Xavier/Glorot initialisation

    def encode(self, input_ids):
        S = input_ids.size(1)
        src_allowed = (input_ids != self.pad_id)[:, None, None, :]          # no attention to padding
        position_bias = self.enc_pos_bias(S, S, input_ids.device)
        x = self.dropout(self.shared_emb(input_ids))
        for layer in self.enc_layers:
            x = layer(x, position_bias, src_allowed)
        return self.dropout(self.enc_norm(x))

    def decode(self, decoder_input_ids, enc_out, input_ids, last_only=False):
        T = decoder_input_ids.size(1)
        src_allowed = (input_ids != self.pad_id)[:, None, None, :]
        causal_allowed = torch.tril(torch.ones(T, T, dtype=torch.bool, device=enc_out.device))[None, None]
        position_bias = self.dec_pos_bias(T, T, enc_out.device)
        x = self.dropout(self.shared_emb(decoder_input_ids))
        for layer in self.dec_layers:
            x = layer(x, enc_out, position_bias, causal_allowed, src_allowed)
        x = self.dropout(self.dec_norm(x))
        if last_only:
            x = x[:, -1:, :]
        return x @ self.shared_emb.weight.t()                                # weight sharing

    def forward(self, input_ids, decoder_input_ids, labels=None):
        enc_out = self.encode(input_ids)
        logits = self.decode(decoder_input_ids, enc_out, input_ids)
        loss = None
        if labels is not None:
            # decoder_input_ids are already shifted (<s> + target[:-1]); -100 = padding, ignored
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), labels.reshape(-1), ignore_index=-100)
        return {"loss": loss, "logits": logits}

### 8.5 The retrieval model (bi-encoder) and the adaptation methods

* **Case vector:** each 512-token chunk goes through the Phase-1 model (decoder-only: causal Transformer / LSTM; T5: encoder only); the chunk vector is the mean of its token states; the case vector is the mean of its chunk vectors; all vectors are L2-normalised, so the dot product is the cosine similarity.
* **Contrastive loss (InfoNCE, as in DPR):** for each query the logits are `similarity / TEMPERATURE` to its relevant candidate, the other candidates of the batch and the hard negatives; cross-entropy with the relevant one as the target.
* **full / partial / LoRMA:** as in the earlier phases.
* **IA3** (Liu et al. 2022): learned vectors that rescale the keys, the values and the inner FFN activations (`k * l_k`, `v * l_v`, `h_ff * l_ff`), initialised to 1, everything else frozen.
* **Prefix tuning** (Module 4.1): trainable key/value prefixes at every layer (through the attention's KV-cache input), reparametrised by an MLP.
* **KD:** the student's scores over the batch candidates are pulled towards the teacher's softened score distribution (score / response-based distillation).

In [ ]:
import re
from safetensors.torch import load_file

def build_base(model_type, n_layers=None):
  V = len(bpe.itos)
  if model_type == "gpt":
    return DecoderLM(LMConfig(vocab_size=V, d_model=N_EMBD, n_layers=n_layers or N_LAYER, n_heads=N_HEAD,
                              n_kv_heads=N_HEAD, d_ff=D_FF, max_seq_len=MODEL_CONTEXT_LENGTH, dropout=DROPOUT,
                              pos_emb_type="learned", norm_type="layernorm", ffn_type="gelu"))
  if model_type == "modern":
    return DecoderLM(LMConfig(vocab_size=V, d_model=N_EMBD, n_layers=n_layers or N_LAYER, n_heads=N_HEAD,
                              n_kv_heads=N_KV_HEAD, d_ff=D_FF, max_seq_len=MODEL_CONTEXT_LENGTH, dropout=DROPOUT,
                              pos_emb_type="rope", norm_type="rmsnorm", ffn_type="swiglu"))
  if model_type == "rnn":
    return RNNLM(V, RNN_HIDDEN, n_layers or RNN_LAYERS, cell_type=RNN_CELL, dropout=DROPOUT)
  if model_type == "t5":
    return T5Model(V + T5_N_SENTINELS, bpe.pad_id, T5_D_MODEL, n_layers or T5_ENC_LAYERS, T5_DEC_LAYERS,
                   T5_HEADS, T5_D_FF, dropout=DROPOUT)
  raise ValueError(model_type)

def hidden_size(model_type):
  return {"gpt": N_EMBD, "modern": N_EMBD, "rnn": RNN_HIDDEN, "t5": T5_D_MODEL}[model_type]

def load_phase1_weights(base, model_type, n_layers=None):
  path = PHASE1_MODEL_FILES[model_type]
  state = load_file(path)
  if n_layers is not None:          # student: keep only the first n_layers layers
    kept = {}
    for k, v in state.items():
      m = re.match(r"^(blocks|cells|enc_layers)\.(\d+)\.", k)
      if m and int(m.group(2)) >= n_layers:
        continue
      kept[k] = v
    state = kept
  missing, unexpected = base.load_state_dict(state, strict=False)
  print(f"loaded Phase-1 {model_type} weights from {path} | missing {len(missing)} | unexpected {len(unexpected)}")
  if missing:
    raise ValueError(f"Phase-1 weights do not match the architecture, missing: {missing[:5]}")

def lm_hidden(base, input_ids):
  # DecoderLM.forward without the LM output layer
  B, T = input_ids.shape
  pos = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, T)
  x = base.tok_emb(input_ids)
  if base.cfg.pos_emb_type == "learned":
    x = x + base.pos_emb(pos)
  x = base.drop(x)
  for block in base.blocks:
    x, _ = block(x, pos)
  return base.ln_f(x)

def rnn_hidden(base, input_ids):
  # RNNLM.forward without the LM output layer
  B, T = input_ids.shape
  x = base.dropout(base.tok_emb(input_ids))
  for l in range(base.num_layers):
    cell = base.cells[l]
    h = torch.zeros(B, base.hidden_size, device=x.device, dtype=x.dtype)
    state = (h, torch.zeros_like(h)) if base.is_lstm else h
    outputs = []
    for t in range(T):
      state = cell(x[:, t, :], state)
      outputs.append(state[0] if base.is_lstm else state)
    x = base.dropout(torch.stack(outputs, dim=1))
  return x


class LoRALinear(nn.Module):
  """LoRA: h = W x + b + scale * B A x  (W, b frozen); scale = alpha / r (LoRA) or alpha / sqrt(r) (rsLoRA)"""
  def __init__(self, base_linear, r, alpha, rank_stabilized=False):
    super().__init__()
    self.base = base_linear
    self.scale = alpha / math.sqrt(r) if rank_stabilized else alpha / r
    self.A = nn.Parameter(torch.randn(r, base_linear.in_features) / math.sqrt(base_linear.in_features))
    self.B = nn.Parameter(torch.zeros(base_linear.out_features, r))

  def forward(self, x):
    return self.base(x) + self.scale * ((x @ self.A.t()) @ self.B.t())


class RSLoRALinear(LoRALinear):
  def __init__(self, base_linear, r, alpha):
    super().__init__(base_linear, r, alpha, rank_stabilized=True)


class LoRMALinear(nn.Module):
  """LoRMA (additive identity): h = (I + (alpha / r) * B A) W x + b  (W, b frozen)"""
  def __init__(self, base_linear, r, alpha):
    super().__init__()
    self.base = base_linear
    self.scale = alpha / r
    self.A = nn.Parameter(torch.randn(r, base_linear.out_features) / math.sqrt(base_linear.out_features))
    self.B = nn.Parameter(torch.zeros(base_linear.out_features, r))

  def forward(self, x):
    y = F.linear(x, self.base.weight)                      # W x
    y = y + self.scale * ((y @ self.A.t()) @ self.B.t())   # (I + s B A) W x
    if self.base.bias is not None:
      y = y + self.base.bias
    return y



def add_adapters(module, adapter_cls, r, alpha):
  # replace every nn.Linear inside the Phase-1 model by an adapter around it
  for name, child in module.named_children():
    if isinstance(child, nn.Linear):
      setattr(module, name, adapter_cls(child, r, alpha))
    else:
      add_adapters(child, adapter_cls, r, alpha)




class IA3Out(nn.Module):
  """IA3 on the output of a linear layer: y = (W x) * l, l starts at 1"""
  def __init__(self, base_linear):
    super().__init__()
    self.base = base_linear
    self.ia3 = nn.Parameter(torch.ones(base_linear.out_features))

  def forward(self, x):
    return self.base(x) * self.ia3


class IA3In(nn.Module):
  """IA3 on the inner FFN activation (the input of the down projection): y = W (h * l)"""
  def __init__(self, base_linear):
    super().__init__()
    self.base = base_linear
    self.ia3 = nn.Parameter(torch.ones(base_linear.in_features))

  def forward(self, x):
    return self.base(x * self.ia3)


def add_ia3(base, model_type):
  if model_type == "rnn":
    raise ValueError("IA3 is defined for the Transformer models (gpt, modern, t5)")
  for m in base.modules():
    if isinstance(m, (CausalSelfAttention, T5Attention)):
      m.k_proj = IA3Out(m.k_proj)
      m.v_proj = IA3Out(m.v_proj)
    if isinstance(m, FeedForward):
      m.net[2] = IA3In(m.net[2])
    if isinstance(m, SwiGLUFeedForward):
      m.w2 = IA3In(m.w2)
    if isinstance(m, T5FeedForward):
      m.fc2 = IA3In(m.fc2)


class PrefixKV(nn.Module):
  """prefix tuning: P trainable positions -> MLP -> keys and values for every layer"""
  def __init__(self, cfg, n_prefix, hidden):
    super().__init__()
    self.P, self.n_layers, self.n_kv, self.d_head = n_prefix, cfg.n_layers, cfg.n_kv_heads, cfg.d_model // cfg.n_heads
    self.prefix_emb = nn.Parameter(torch.randn(n_prefix, cfg.d_model) * 0.02)
    self.mlp = nn.Sequential(nn.Linear(cfg.d_model, hidden), nn.Tanh(),
                             nn.Linear(hidden, self.n_layers * 2 * self.n_kv * self.d_head))

  def forward(self, B):
    x = self.mlp(self.prefix_emb).view(self.P, self.n_layers, 2, self.n_kv, self.d_head).permute(1, 2, 3, 0, 4)
    return [(x[l, 0].unsqueeze(0).expand(B, -1, -1, -1), x[l, 1].unsqueeze(0).expand(B, -1, -1, -1))
            for l in range(self.n_layers)]


def decoder_hidden(base, input_ids, prefix=None):
  # DecoderLM.forward without the output layer; with prefix tuning the learned keys/values come in as the KV cache
  B, T = input_ids.shape
  pos = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, T)
  x = base.tok_emb(input_ids)
  if base.cfg.pos_emb_type == "learned":
    x = x + base.pos_emb(pos)
  x = base.drop(x)
  past = prefix(B) if prefix is not None else [None] * len(base.blocks)
  mask = torch.ones(B, prefix.P + T, dtype=torch.long, device=input_ids.device) if prefix is not None else None
  for block, layer_past in zip(base.blocks, past):
    x, _ = block(x, pos, mask, layer_past)
  return base.ln_f(x)


class RetrievalModel(nn.Module):
  def __init__(self, base, model_type, prefix=None):
    super().__init__()
    self.base = base
    self.model_type = model_type
    self.prefix = prefix

  def token_states(self, ids):
    if self.model_type == "t5":
      return self.base.encode(ids)
    if self.model_type == "rnn":
      return rnn_hidden(self.base, ids)
    return decoder_hidden(self.base, ids, self.prefix)

  def encode(self, ids):
    """ids (B, N, T) -> case vectors (B, D), chunk vectors (B, N, D), which chunks exist (B, N)"""
    B, N, T = ids.shape
    flat = ids.view(B * N, T)
    real = (flat != bpe.pad_id).any(dim=-1)
    h = self.token_states(flat[real])
    m = (flat[real] != bpe.pad_id)
    if TOKEN_POOL == "last":
      last = m.sum(dim=1) - 1
      v = h[torch.arange(h.size(0), device=h.device), last]
    else:
      mf = m.unsqueeze(-1).to(h.dtype)
      v = (h * mf).sum(dim=1) / mf.sum(dim=1).clamp(min=1)
    v = F.normalize(v.float(), dim=-1)
    chunks = v.new_zeros(B * N, v.size(-1))
    chunks[real] = v
    chunks = chunks.view(B, N, -1)
    present = real.view(B, N)
    p = present.unsqueeze(-1).float()
    doc = F.normalize((chunks * p).sum(dim=1) / p.sum(dim=1).clamp(min=1), dim=-1)
    return doc, chunks, present

  @staticmethod
  def similarity(q, c):
    """q, c = outputs of encode -> (n_queries, n_candidates) similarity matrix"""
    if SIMILARITY == "cosine":
      return q[0] @ c[0].T
    s = torch.einsum("qnd,cmd->qcnm", q[1], c[1])                              # chunk-to-chunk cosines
    s = s.masked_fill(~c[2][None, :, None, :], -1e4).max(dim=-1).values         # best candidate chunk
    qp = q[2][:, None, :].float()
    return (s * qp).sum(dim=-1) / qp.sum(dim=-1).clamp(min=1)                  # mean over the query's chunks

  def scores(self, q_ids, p_ids, n_ids=None):
    q = self.encode(q_ids)
    cand_ids = p_ids if n_ids is None else torch.cat([p_ids, n_ids], dim=0)
    c = self.encode(cand_ids)
    return self.similarity(q, c) / TEMPERATURE              # (B, B) or (B, 2B); the relevant one of query i is column i

  def forward(self, q_ids, p_ids, n_ids=None):
    logits = self.scores(q_ids, p_ids, n_ids)
    target = torch.arange(logits.size(0), device=logits.device)
    return {"loss": F.cross_entropy(logits, target)}


def set_trainable(model, ft_method):
  base = model.base
  for name, p in base.named_parameters():
    if ft_method == "full":
      p.requires_grad = True
    elif ft_method == "lorma":
      p.requires_grad = name.endswith(".A") or name.endswith(".B")
    elif ft_method == "ia3":
      p.requires_grad = name.endswith(".ia3")
    else:
      p.requires_grad = False
  if ft_method == "partial":
    if model.model_type in ("gpt", "modern"):
      layers, final_norm = base.blocks, base.ln_f
    elif model.model_type == "rnn":
      layers, final_norm = base.cells, None
    else:
      layers, final_norm = base.enc_layers, base.enc_norm
    for layer in list(layers)[-PARTIAL_TOP_LAYERS:]:
      for p in layer.parameters():
        p.requires_grad = True
    if final_norm is not None:
      for p in final_norm.parameters():
        p.requires_grad = True
  if model.prefix is not None:
    for p in model.prefix.parameters():
      p.requires_grad = True


def build_pcr_model(model_type, ft_method, n_layers=None, load_pretrained=True):
  base = build_base(model_type, n_layers)
  if load_pretrained:
    load_phase1_weights(base, model_type, n_layers)
  if model_type == "t5":
    base.dec_layers = nn.ModuleList()       # only the T5 encoder is used
  prefix = None
  if ft_method == "lorma":
    add_adapters(base, LoRMALinear, ADAPTER_RANK, ADAPTER_ALPHA)
  elif ft_method == "ia3":
    add_ia3(base, model_type)
  elif ft_method == "prefix":
    if model_type not in ("gpt", "modern"):
      raise ValueError("prefix tuning is implemented for the decoder-only Transformers (gpt, modern)")
    prefix = PrefixKV(base.cfg, PREFIX_LENGTH, PREFIX_HIDDEN)
  model = RetrievalModel(base, model_type, prefix)
  set_trainable(model, ft_method)
  return model


class KDRetrieval(nn.Module):
  """loss = alpha * contrastive(gold) + (1 - alpha) * T^2 * KL(softmax(teacher/T) || softmax(student/T)) over the batch candidates"""
  def __init__(self, student, teacher, temperature, alpha):
    super().__init__()
    self.student = student
    self.teacher = teacher
    self.T = temperature
    self.alpha = alpha
    for p in self.teacher.parameters():
      p.requires_grad = False

  def train(self, mode=True):
    super().train(mode)
    self.teacher.eval()
    return self

  def forward(self, q_ids, p_ids, n_ids=None):
    s = self.student.scores(q_ids, p_ids, n_ids).float()
    with torch.no_grad():
      t = self.teacher.scores(q_ids, p_ids, n_ids).float()
    target = torch.arange(s.size(0), device=s.device)
    gold = F.cross_entropy(s, target)
    kd = F.kl_div(F.log_softmax(s / self.T, dim=-1), F.softmax(t / self.T, dim=-1), reduction="batchmean") * self.T * self.T
    return {"loss": self.alpha * gold + (1 - self.alpha) * kd}


if USE_KD:
  pcr_model = build_pcr_model(MODEL_TYPE, FT_METHOD, n_layers=STUDENT_LAYERS)
  teacher = build_pcr_model(TEACHER_MODEL_TYPE, TEACHER_FT_METHOD, load_pretrained=False)
  teacher.load_state_dict(torch.load(TEACHER_PCR_PATH, map_location="cpu"))
  print("teacher loaded from", TEACHER_PCR_PATH)
  model = KDRetrieval(pcr_model, teacher, KD_TEMPERATURE, KD_ALPHA)
  n_teacher = sum(p.numel() for p in teacher.parameters())
else:
  pcr_model = build_pcr_model(MODEL_TYPE, FT_METHOD, load_pretrained=USE_PHASE1_WEIGHTS)
  model = pcr_model
  n_teacher = 0
n_total = sum(p.numel() for p in pcr_model.parameters())
n_trainable = sum(p.numel() for p in pcr_model.parameters() if p.requires_grad)
print(f"retriever: {n_total / 1e6:.2f}M parameters | trainable {n_trainable / 1e6:.3f}M ({100 * n_trainable / n_total:.2f}%)")
if USE_KD:
  print(f"teacher: {n_teacher / 1e6:.2f}M parameters -> student is {100 * n_total / n_teacher:.0f}% of the teacher")

## 9. Retrieval with the model, and the zero-shot Phase-1 score

Every candidate of a pool is encoded once (an index of case vectors); a query is compared with all of them. Before training, the **zero-shot** Phase-1 model (LM weights only, adapters at their identity start) is scored on dev: this shows how well the Phase-1 representations transfer to retrieval without any retrieval training.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = torch.cuda.is_available() and MODEL_TYPE != "t5"

def encode_cases(m, cases):
  m.eval()
  docs, chunks, present = [], [], []
  with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
    for s in range(0, len(cases), ENCODE_BATCH):
      ids = doc_tensor([c["row"] for c in cases[s:s + ENCODE_BATCH]]).to(device)
      d, ch, pr = m.encode(ids)
      docs.append(d.float().cpu()); chunks.append(ch.float().cpu()); present.append(pr.cpu())
  return torch.cat(docs), torch.cat(chunks), torch.cat(present)

def model_scores(m, queries, pool):
  t0 = time.time()
  c = encode_cases(m, pool)
  q = encode_cases(m, queries)
  enc_seconds = time.time() - t0
  out = []
  for s in range(0, len(queries), 64):
    qs = (q[0][s:s + 64], q[1][s:s + 64], q[2][s:s + 64])
    out.append(RetrievalModel.similarity(qs, c).numpy())
  return np.concatenate(out), enc_seconds / (len(queries) + len(pool)), c[0].shape[1]

def zscore(s):
  return (s - s.mean(axis=1, keepdims=True)) / (s.std(axis=1, keepdims=True) + 1e-9)

def fuse(neural, bm25, w):
  return w * zscore(neural) + (1 - w) * zscore(bm25)

zero_shot = None
if ZERO_SHOT_EVAL and not LOAD_PCR_PATH:
  pcr_model.to(device)
  s0, _, _ = model_scores(pcr_model, dev_q, dev_pool)
  zero_shot = evaluate(rank(s0, dev_q, dev_pool), dev_q, dev_pool)
  show("DEV  zero-shot Phase-1 model (no training)", zero_shot)
  del s0

## 10. Training (Trainer, like the earlier phases)

Checkpoint selection by the validation contrastive loss on dev triples.

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
  output_dir=os.path.join(EXP_DIR, "checkpoints"),
  num_train_epochs=EPOCHS,
  per_device_train_batch_size=BATCH_SIZE,
  per_device_eval_batch_size=BATCH_SIZE,
  gradient_accumulation_steps=GRAD_ACCUM,
  learning_rate=LEARNING_RATE,
  weight_decay=0.01,
  warmup_steps=5 if SMOKE_TEST else 30,
  lr_scheduler_type="cosine",
  max_grad_norm=1.0,
  fp16=torch.cuda.is_available() and MODEL_TYPE != "t5",
  eval_strategy="epoch",
  save_strategy="epoch",
  save_total_limit=1,
  load_best_model_at_end=True,
  metric_for_best_model="eval_loss",
  greater_is_better=False,
  prediction_loss_only=True,
  dataloader_drop_last=True,           # every GPU gets the same number of triples (in-batch negatives)
  logging_steps=10,
  seed=SEED,
  report_to="none",
  remove_unused_columns=False,
  label_names=["p_ids"],               # the target is the position of the relevant candidate; this only tells the Trainer to report the validation loss
)

if torch.cuda.is_available():
  torch.cuda.reset_peak_memory_stats()
start_time = time.time()
if LOAD_PCR_PATH:
  pcr_model.load_state_dict(torch.load(LOAD_PCR_PATH, map_location="cpu"))
  print("loaded trained retriever from", LOAD_PCR_PATH, "- skipping training")
  log_history = []
elif SKIP_TRAINING:
  print("SKIP_TRAINING: the zero-shot Phase-1 model is the final retriever")
  log_history = []
else:
  trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=val_dataset,
                    data_collator=data_collator)
  trainer.train()
  log_history = trainer.state.log_history
training_time = time.time() - start_time
peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
print(f"Training time: {training_time / 60:.1f} min | peak GPU memory: {peak_memory_gb:.2f} GB")
for h in log_history:
  if "eval_loss" in h:
    print(f"epoch {h['epoch']:.2f}: validation contrastive loss {h['eval_loss']:.4f}")
  elif "loss" in h and int(h.get("step", 0)) % 50 == 0:
    print(f"   step {h['step']}: train loss {h['loss']:.4f}")

## 11. Dev evaluation and analysis

* micro P / R / F1 at 7 (the metric) and at other k (precision-recall trade-off), recall@100;
* BM25 vs model vs fusion (weight tuned on dev), and how much their top-7 lists overlap;
* errors: queries with no relevant case in the top 7, rank of the first relevant case, F1 by query length and by number of relevant cases;
* cost: parameters, encoding time per case, index size.

In [ ]:
pcr_model.to(device)
dev_scores, enc_s_per_case, dim = model_scores(pcr_model, dev_q, dev_pool)
order_m = rank(dev_scores, dev_q, dev_pool)
ev_model = evaluate(order_m, dev_q, dev_pool, ks=(1, 3, 5, 7, 10, 20, 50, 100))
print(f"DEV ({len(dev_q)} queries, pool {len(dev_pool)}):")
show("BM25 (no model)", ev_bm25)
if zero_shot:
  show("zero-shot Phase-1 model", zero_shot)
show("trained model", ev_model)
best_w, best_f1 = 1.0, ev_model[TOP_K]["F1"]
fusion_curve = {}
for w in [i / 10 for i in range(11)]:
  f1 = evaluate(rank(fuse(dev_scores, bm25_dev, w), dev_q, dev_pool), dev_q, dev_pool)[TOP_K]["F1"]
  fusion_curve[w] = round(f1, 4)
  if w >= 0.1 and f1 > best_f1:              # w = 0 would be BM25 alone (no model), so the fused submission keeps >= 10% model
    best_w, best_f1 = w, f1
print("fusion F1@7 by model weight (0 = BM25 only, 1 = model only):", fusion_curve)
ev_fused = evaluate(rank(fuse(dev_scores, bm25_dev, best_w), dev_q, dev_pool), dev_q, dev_pool, ks=(1, 3, 5, 7, 10, 20, 50, 100))
show(f"BM25 + model fusion (model weight {best_w})", ev_fused)

print("\nprecision / recall / F1 at k (trained model | BM25):")
for k in (1, 3, 5, 7, 10, 20):
  a, b = ev_model[k], ev_bm25[k]
  print(f"   k={k:2d}: model P {a['P']:.3f} R {a['R']:.3f} F1 {a['F1']:.3f} | BM25 P {b['P']:.3f} R {b['R']:.3f} F1 {b['F1']:.3f}")
print(f"recall@100: model {ev_model[100]['R']:.3f} | BM25 {ev_bm25[100]['R']:.3f} | fusion {ev_fused[100]['R']:.3f}")

order_b = rank(bm25_dev, dev_q, dev_pool)
overlap = np.mean([len(set(order_m[i, :TOP_K]) & set(order_b[i, :TOP_K])) / TOP_K for i in range(len(dev_q))])
print(f"top-7 overlap between the model and BM25: {100 * overlap:.0f}%")

per_f1, first_rank, n_rel, qlen = [], [], [], []
for i, q in enumerate(dev_q):
  rel = set(q["relevant"])
  top = [dev_pool[j]["id"] for j in order_m[i, :TOP_K]]
  h = sum(1 for c in top if c in rel)
  p, r = h / TOP_K, h / max(1, len(rel))
  per_f1.append(2 * p * r / (p + r) if p + r else 0.0)
  ranks = [k for k, j in enumerate(order_m[i]) if dev_pool[j]["id"] in rel]
  first_rank.append(ranks[0] + 1 if ranks else len(dev_pool))
  n_rel.append(len(rel))
  qlen.append(DOC_LEN[q["row"]])
per_f1, first_rank = np.array(per_f1), np.array(first_rank)
print(f"\nqueries with no relevant case in the top 7: {100 * np.mean(per_f1 == 0):.0f}% | median rank of the first relevant case: {int(np.median(first_rank))}")
for name, idx in zip(["short", "medium", "long"], np.array_split(np.argsort(qlen), 3)):
  print(f"   {name:6s} queries ({len(idx)}): mean F1@7 {per_f1[idx].mean():.3f} | chunks cover {100 * np.mean([min(1, N_CHUNKS * CHUNK_TOKENS / qlen[i]) for i in idx]):.0f}% of the query")
for name, idx in zip(["few", "medium", "many"], np.array_split(np.argsort(n_rel), 3)):
  print(f"   {name:6s} relevant cases ({min(n_rel[i] for i in idx)}-{max(n_rel[i] for i in idx)}): mean F1@7 {per_f1[idx].mean():.3f}")
print(f"\ncost: {n_total / 1e6:.1f}M parameters | encoding {1000 * enc_s_per_case:.0f} ms per case | index: {dim}-dim vectors, "
      f"{len(test_pool) * dim * 4 / 1e6:.1f} MB for the test pool (float32)")

## 12. Save the retriever (needed when this experiment is the KD teacher)

In [ ]:
if LOAD_PCR_PATH or SKIP_TRAINING:
  print("no training in this run: model not saved")
else:
  torch.save(pcr_model.state_dict(), os.path.join(EXP_DIR, "pcr_model.pt"))
  print("saved", os.path.join(EXP_DIR, "pcr_model.pt"))

## 13. Test ranking -> prediction.json -> submission zips

`{query_id: [7 candidate ids]}`, best first, unique, only from `test_candidates.json`, never the query itself. Two zips: the model alone, and BM25 + model fusion with the dev-tuned weight.

In [ ]:
t0 = time.time()
test_scores, _, _ = model_scores(pcr_model, test_q, test_pool)
test_seconds = time.time() - t0
pool_ids = set(c["id"] for c in test_pool)

def write_submission(scores, tag):
  order = rank(scores, test_q, test_pool)
  pred = {}
  for i, q in enumerate(test_q):
    top = [test_pool[j]["id"] for j in order[i, :TOP_K]]
    assert len(set(top)) == TOP_K and all(c in pool_ids for c in top) and q["id"] not in top
    pred[q["id"]] = top
  assert len(pred) == len(test_q)
  path = os.path.join(EXP_DIR, f"prediction_{tag}.json")
  with open(path, "w") as f:
    json.dump(pred, f)
  zpath = os.path.join(OUTPUT_DIR, f"submission_{EXPERIMENT_NAME}_{tag}.zip")
  with zipfile.ZipFile(zpath, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(path, arcname="prediction.json")
  print("submission:", zpath)
  return pred

# save the raw scores so that several runs can be combined later (ensemble notebook)
np.savez_compressed(os.path.join(EXP_DIR, "scores.npz"), dev=dev_scores.astype(np.float32), test=test_scores.astype(np.float32),
                    bm25_dev=bm25_dev.astype(np.float32), bm25_test=bm25_test.astype(np.float32),
                    dev_q=np.array([q["id"] for q in dev_q]), test_q=np.array([q["id"] for q in test_q]),
                    dev_pool=np.array([c["id"] for c in dev_pool]), test_pool=np.array([c["id"] for c in test_pool]),
                    dev_rel=np.array([json.dumps(q["relevant"]) for q in dev_q]))
print("saved", os.path.join(EXP_DIR, "scores.npz"))
pred_model = write_submission(test_scores, "model")
if FUSION:
  pred_fused = write_submission(fuse(test_scores, bm25_test, best_w), "fused")
print(f"{len(test_q)} test queries ranked against {len(test_pool)} candidates in {test_seconds:.0f}s")

## 14. Log this experiment (one line per experiment in results.jsonl)

In [ ]:
result = {
  "experiment": EXPERIMENT_NAME, "model": MODEL_TYPE, "method": FT_METHOD, "kd": USE_KD,
  "student_layers": STUDENT_LAYERS if USE_KD else None,
  "teacher": f"{TEACHER_MODEL_TYPE}/{TEACHER_FT_METHOD}" if USE_KD else None,
  "phase1_weights": USE_PHASE1_WEIGHTS, "n_chunks": N_CHUNKS, "chunk_mode": CHUNK_MODE, "token_pool": TOKEN_POOL,
  "similarity": SIMILARITY, "negatives": NEGATIVES, "train_triples": len(train_dataset),
  "params_M": round(n_total / 1e6, 2), "trainable_M": round(n_trainable / 1e6, 3),
  "trainable_pct": round(100 * n_trainable / n_total, 2),
  "train_min": round(training_time / 60, 1), "peak_mem_GB": round(peak_memory_gb, 2),
  "encode_ms_per_case": round(1000 * enc_s_per_case, 1), "dim": int(dim),
  "dev_bm25_F1@7": round(ev_bm25[TOP_K]["F1"], 4),
  "dev_zeroshot_F1@7": round(zero_shot[TOP_K]["F1"], 4) if zero_shot else None,
  "dev_model_F1@7": round(ev_model[TOP_K]["F1"], 4), "dev_model_P@7": round(ev_model[TOP_K]["P"], 4),
  "dev_model_R@7": round(ev_model[TOP_K]["R"], 4), "dev_model_R@100": round(ev_model[100]["R"], 4),
  "dev_fused_F1@7": round(ev_fused[TOP_K]["F1"], 4), "fusion_weight": best_w,
  "lr": LEARNING_RATE, "epochs": EPOCHS,
}
with open(RESULTS_PATH, "a") as f:
  f.write(json.dumps(result) + "\n")
print(json.dumps(result, indent=1))